In [1]:
import os, csv, json, random
from datetime import datetime, timedelta

BASE_DIR = r"C:\Users\yashi\OneDrive\Desktop\Project_TCS"
directories = [
    os.path.join(BASE_DIR, "source_social_media"),
    os.path.join(BASE_DIR, "source_ecommerce"),
    os.path.join(BASE_DIR, "source_support_center")
]
for d in directories:
    os.makedirs(d, exist_ok=True)

start_date = datetime(2026, 5, 1)
num_days = 120                 # <-- extend from 30 -> 120
base_daily_volume = 15

def add_anomaly_day(day_idx):
    # Put anomalies at a few points (edit these)
    # day_idx is 0-based: day_idx=0 => start_date
    return day_idx in {18, 55, 92}

def anomaly_category(day_idx):
    # Which category “emerges” on each anomaly window
    # edit/add as needed
    if day_idx == 18:
        return "Shipping"
    if day_idx == 55:
        return "Inventory"
    if day_idx == 92:
        return "Technology"
    return None

# Split fragments into “good” vs “bad” so anomaly windows are more negative
fragments = {
    "Staff": {
        "good": {
            "subjects": [
                "The cashier at the front counter",
                "The floor staff working today",
                "The customer service representative",
                "The associate handling my checkout",
            ],
            "actions": [
                "was incredibly polite and welcoming",
                "helped me find everything instantly",
                "handled a tough situation beautifully",
                "was working at an average pace",
                "processed my request routinely",
            ],
            "endings": [
                "definitely coming back because of them!",
                "highly recommend this location.",
                "with no issues to report.",
                "just a normal, standard interaction.",
            ],
        },
        "bad": {
            "subjects": [
                "The cashier at the front counter",
                "The floor staff working today",
                "The manager on duty",
                "The associate handling my checkout",
            ],
            "actions": [
                "completely ignored me",
                "glared at me aggressively",
                "seemed completely untrained and lost",
                "stood around talking to coworkers",
                "was working at a slow pace",
            ],
            "endings": [
                "which made the entire experience miserable.",
                "while I stood waiting for twenty minutes.",
                "leaving me frustrated the entire time.",
                "definitely not returning because of them.",
            ],
        }
    },
    "Technology": {
        "good": {
            "subjects": [
                "The digital checkout screen",
                "The mobile app cart page",
                "The self-serve kiosk",
                "The online payment gateway",
            ],
            "actions": [
                "worked perfectly without a single hitch",
                "processed my transaction in under two seconds",
                "is incredibly intuitive and clean",
                "accepted the standard tap payment process",
            ],
            "endings": [
                "nothing out of the ordinary for this system.",
                "completed the task as expected.",
                "super smooth user experience overall.",
            ],
        },
        "bad": {
            "subjects": [
                "The digital checkout screen",
                "The mobile app cart page",
                "The self-serve kiosk",
                "The online payment gateway",
            ],
            "actions": [
                "kept spinning on an infinite loading loop",
                "crashed completely and shut down",
                "refused to read my barcode input",
                "would not let me proceed",
            ],
            "endings": [
                "and wouldn't let me hit the confirm button.",
                "forcing me to restart my device entirely.",
                "leaving me stuck at checkout.",
            ],
        }
    },
    "Inventory": {
        "good": {
            "subjects": [
                "The physical shelves for the new collection",
                "The stock levels for advertised sale items",
                "The online availability tracker",
                "The main display in the center aisle",
            ],
            "actions": [
                "were beautifully organized and full",
                "matched the digital inventory perfectly",
                "were fully stocked with fresh products",
                "had a few standard options available",
            ],
            "endings": [
                "lots of great variety to choose from today.",
                "which is fine for a routine shopping trip.",
                "found exactly what I came for.",
            ],
        },
        "bad": {
            "subjects": [
                "The physical shelves for the new collection",
                "The stock levels for advertised sale items",
                "The online availability tracker",
            ],
            "actions": [
                "were completely cleared out and bare",
                "looked totally disorganized and messy",
                "had absolutely nothing left in stock",
                "did not match what was advertised",
            ],
            "endings": [
                "making the long drive out here a total waste of time.",
                "why bother advertising a major sale if you have no stock?",
                "leaving me disappointed and empty-handed.",
            ],
        }
    },
    "Shipping": {
        "good": {
            "subjects": [
                "The package tracking number",
                "The cardboard shipping box",
                "The standard delivery transit timeframe",
                "The order confirmation system",
            ],
            "actions": [
                "updated tracking alerts in real-time",
                "arrived incredibly fast and ahead of schedule",
                "kept the items perfectly safe and secure",
                "updated status on the third business day",
            ],
            "endings": [
                "could not ask for a better delivery experience.",
                "will absolutely order online again.",
                "standard shipping experience overall.",
            ],
        },
        "bad": {
            "subjects": [
                "The package tracking number",
                "The cardboard shipping box",
                "The standard delivery transit timeframe",
                "The order confirmation system",
            ],
            "actions": [
                "hasn't updated status a single time in weeks",
                "took fourteen days longer than promised",
                "arrived completely crushed and smashed",
                "didn't show updated tracking",
            ],
            "endings": [
                "and my credit card was already billed.",
                "leaving the fragile contents inside totally broken.",
                "forcing me to contact support repeatedly.",
            ],
        }
    },
}

def generate_blind_text(category, sentiment_bucket):
    # sentiment_bucket: "good" or "bad"
    cfg = fragments[category][sentiment_bucket]
    subj = random.choice(cfg["subjects"])
    action = random.choice(cfg["actions"])
    ending = random.choice(cfg["endings"])
    return f"{subj} {action} {ending}"

social_records = []
ecomm_records = []
support_records = []

customer_id_pool = [f"{i}" for i in range(1000, 1400)]

# channel weights (edit if you want)
channel_weights = [("social", 0.35), ("ecomm", 0.35), ("support", 0.30)]

for day in range(num_days):
    current_date = start_date + timedelta(days=day)

    anomaly = add_anomaly_day(day)
    daily_volume = base_daily_volume

    # Base volume + anomaly volume spike
    if anomaly:
        daily_volume = int(base_daily_volume * 6)  # <-- spike size

    # pick anomaly category if in anomaly window
    emergent_cat = anomaly_category(day) if anomaly else None

    for _ in range(random.randint(max(1, daily_volume - 6), daily_volume + 6)):
        timestamp = current_date + timedelta(hours=random.randint(8, 20), minutes=random.randint(0, 59))
        customer_id = random.choice(customer_id_pool)

        # During anomaly day, category strongly shifts to emergent_cat
        if emergent_cat and random.random() < 0.85:
            category = emergent_cat
        else:
            category = random.choice(["Staff", "Technology", "Inventory", "Shipping"])

        # During anomaly day, sentiment flips to be more negative (so models see “emerging issues”)
        if emergent_cat and category == emergent_cat and random.random() < 0.80:
            sentiment_bucket = "bad"
        else:
            sentiment_bucket = "good" if random.random() < 0.7 else "bad"

        text = generate_blind_text(category, sentiment_bucket)

        channel = random.choices(
            population=[c for c, _ in channel_weights],
            weights=[w for _, w in channel_weights],
            k=1
        )[0]

        if channel == "social":
            social_records.append([
                timestamp.strftime("%Y-%m-%d %H:%M:%S"),
                f"@user_{random.randint(1000, 9999)}",
                customer_id,
                text
            ])
        elif channel == "ecomm":
            ecomm_records.append({
                "review_date": timestamp.strftime("%d-%b-%Y"),
                "rating": random.randint(1, 5),
                "customer_id": customer_id,
                "body": text
            })
        else:
            support_records.append([
                timestamp.strftime("%Y-%m-%d %I:%M %p"),
                customer_id,
                text
            ])

# Save files
with open(os.path.join(BASE_DIR, "source_social_media", "twitter_dump.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "handle", "customer_id", "tweet_text"])
    writer.writerows(social_records)

with open(os.path.join(BASE_DIR, "source_ecommerce", "web_reviews.json"), "w", encoding="utf-8") as f:
    json.dump(ecomm_records, f, indent=2)

with open(os.path.join(BASE_DIR, "source_support_center", "support_logs.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["date_string", "customer_id", "transcript"])
    writer.writerows(support_records)

print("🔥 Generated synthetic multi-channel feedback with multiple emerging issue windows.")

🔥 Generated synthetic multi-channel feedback with multiple emerging issue windows.
